# Digital Twin Augsburg – Georgsvorstadt
## Explorative Datenanalyse

Dieses Notebook visualisiert die OSM-Gebäudedaten des Quartiers und gibt einen Überblick über Höhenverteilung, Nutzungsklassen und Datenvollständigkeit.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

from configs.settings import BUILDINGS_GEOJSON, DATA_INTERIM

In [ ]:
# Bereinigtes GeoJSON laden (nach Phase 2)
clean_path = DATA_INTERIM / 'georgsvorstadt_clean.geojson'
gdf = gpd.read_file(clean_path)
gdf_metric = gdf.to_crs('EPSG:25832')

print(f'Gebaeude gesamt : {len(gdf)}')
print(f'CRS             : {gdf.crs}')
print(f'Spalten         : {list(gdf.columns)}')
gdf.head(3)

## 1 · Karte der Gebäudegrundrisse

In [ ]:
DEFAULT_H = 9.6  # 3 Geschosse × 3.2 m (Fallback)

fig, ax = plt.subplots(figsize=(12, 10))

# Farbe nach Gebäudehöhe
gdf_metric.plot(
    ax=ax,
    column='height_m',
    cmap='YlOrRd',
    legend=True,
    legend_kwds={'label': 'Hoehe [m]', 'shrink': 0.6},
    edgecolor='grey',
    linewidth=0.3,
)

ax.set_title('Georgsvorstadt – Gebäudegrundrisse (Farbe = Höhe)', fontsize=14)
ax.set_xlabel('Rechtswert (m)')
ax.set_ylabel('Hochwert (m)')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(DATA_INTERIM.parent / 'output' / 'map_heights.png', dpi=150)
plt.show()

## 2 · Höhenverteilung

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramm
gdf['height_m'].plot.hist(ax=axes[0], bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(DEFAULT_H, color='red', linestyle='--', label=f'Fallback {DEFAULT_H} m')
axes[0].set_xlabel('Hoehe [m]')
axes[0].set_ylabel('Anzahl Gebaeude')
axes[0].set_title('Hoehenverteilung')
axes[0].legend()

# Datenvollständigkeit
has_height  = gdf['height'].notna() & (gdf['height'].astype(str).str.strip() != '')
has_levels  = gdf['levels'].notna() & (gdf['levels'].astype(str).str.strip() != '')
fallback    = ~has_height & ~has_levels

labels  = ['height-Tag', 'levels-Tag', 'Fallback (kein Tag)']
values  = [has_height.sum(), (~has_height & has_levels).sum(), fallback.sum()]
colors  = ['#2ecc71', '#f39c12', '#e74c3c']

axes[1].pie(values, labels=labels, colors=colors, autopct='%1.0f%%', startangle=140)
axes[1].set_title('Hoehenquelle')

plt.tight_layout()
plt.show()

print(f'Gesamt          : {len(gdf)}')
print(f'height-Tag      : {has_height.sum()} ({has_height.mean()*100:.0f}%)')
print(f'levels-Tag      : {(~has_height & has_levels).sum()}')
print(f'Fallback {DEFAULT_H} m : {fallback.sum()} ({fallback.mean()*100:.0f}%)')

## 3 · Nutzungsklassen

In [ ]:
building_types = gdf['building'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(10, 4))
building_types.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('OSM building-Tag')
ax.set_ylabel('Anzahl')
ax.set_title('Gebaeudenutzung (OSM building-Tag)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 4 · Grundflächen-Statistik (EPSG:25832)

In [ ]:
gdf_metric['footprint_m2'] = gdf_metric.geometry.area
gdf_metric['levels_num']   = pd.to_numeric(gdf['levels'], errors='coerce').fillna(3)
gdf_metric['gfa_m2']       = gdf_metric['footprint_m2'] * gdf_metric['levels_num']

print('Grundflaeche (Footprint):')
print(gdf_metric['footprint_m2'].describe().round(1).to_string())
print(f'\nGesamte Bruttogeschossflaeche (GFA): {gdf_metric["gfa_m2"].sum():,.0f} m2')